## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import re
import urllib.request
api_url = 'https://raw.githubusercontent.com/tanmayyb/ele70_bv03/refs/heads/main/api/datasets.py'
exec(urllib.request.urlopen(api_url).read())

## Load Datasets and Preprocess

In [2]:
ieso = IESODataset('zonal') # option 1
# ieso = IESODataset('fsa') # option 2

Available years: 2003 to 2024


In [3]:
target_options = ieso.get_target_options() # returns list of the target options
available_dates = ieso.get_dates() # returns list of available dates (str)


print(target_options)
print(available_dates)


['Northwest', 'Northeast', 'Ottawa', 'East', 'Toronto', 'Essa', 'Bruce', 'Southwest', 'Niagara', 'West', 'Zone Total']
[2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [4]:
target_val = 4
ieso.set_target(target_val)
ieso.load_dataset(start_date=2010, end_date=2020, download=True)

climate = ClimateDataset(ieso)
climate.load_dataset(sample_num=5, download=True)

preprocessor = DatasetPreprocessor(ieso, climate)
target_name, dataset = preprocessor.preprocess()

Processing chunks: 100%|██████████| 3/3 [00:00<00:00,  4.29it/s]


In [5]:
dataset

,Toronto,71265_temp,71265_dwpt,71265_rhum,71265_prcp,71265_snow,71265_wdir,71265_wspd,71265_wpgt,71265_pres,...,71508_wdir,71508_wspd,71508_wpgt,71508_pres,71508_tsun,71508_coco,Y,M,D,H
0,4966,2.5,1.3,92.0,1.942053,NaN,230.0,22.3,NaN,1013.7,...,0.0,0.0,NaN,1016.35101,NaN,NaN,2010,1,1,0
1,4761,2.0,1.0,93.0,1.942053,NaN,240.0,16.6,NaN,1013.7,...,0.0,0.0,NaN,1016.35101,NaN,NaN,2010,1,1,1
2,4594,2.0,1.0,93.0,1.942053,NaN,250.0,14.8,NaN,1013.7,...,0.0,0.0,NaN,1016.35101,NaN,NaN,2010,1,1,2
3,4443,2.0,1.0,93.0,1.942053,NaN,250.0,14.8,NaN,1013.6,...,0.0,0.0,NaN,1016.35101,NaN,NaN,2010,1,1,3
4,4375,2.0,1.0,93.0,1.942053,NaN,250.0,14.8,NaN,1013.7,...,0.0,0.0,NaN,1016.35101,NaN,NaN,2010,1,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96333,5948,2.0,-3.9,65.0,1.942053,NaN,270.0,20.5,NaN,1024.2,...,0.0,0.0,NaN,1023.90000,NaN,NaN,2020,12,31,19
96334,5741,2.0,-2.9,70.0,1.942053,NaN,260.0,18.4,NaN,1025.0,...,0.0,0.0,NaN,1024.70000,NaN,NaN,2020,12,31,20
96335,5527,2.1,-3.2,68.0,1.942053,NaN,270.0,18.4,NaN,1025.8,...,0.0,0.0,NaN,1025.40000,NaN,NaN,2020,12,31,21
96336,5301,2.0,-2.9,70.0,1.942053,NaN,270.0,20.5,NaN,1026.7,...,0.0,0.0,NaN,1026.50000,NaN,NaN,2020,12,31,22


In [10]:
def create_train_test_split(dataset:pd.DataFrame, target:str='target', split_coeff:float=0.8, dt=None) -> tuple:
  training_cutoff = int(split_coeff*len(dataset))
  train = dataset.iloc[:training_cutoff]
  test = dataset.iloc[training_cutoff:]

  X_train = train.drop(columns=[target])
  y_train = train[target]
  X_test = test.drop(columns=[target])
  y_test = test[target]

  (train_idx, test_idx) = None, None
  if dt is not None:
    train_idx = dt[:training_cutoff]
    test_idx = dt[training_cutoff:]

  return (X_train, X_test, y_train, y_test), (train_idx, test_idx)

target = target_name

(X_train, X_test, y_train, y_test), (train_idx, test_idx) = create_train_test_split(dataset, target=target, dt=None)
y_test_numpy = y_test.to_numpy()

## XGBoost Regression Model for Time Series Forecasting

In [11]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
from xgboost import plot_importance, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [12]:
# https://www.kaggle.com/code/robikscube/tutorial-time-series-forecasting-with-xgboost#Create-XGBoost-Model
reg = xgb.XGBRegressor(n_estimators=1000)
reg.fit(X_train, y_train,
        eval_set=[(X_train, y_train), (X_test, y_test)],
        # early_stopping_rounds=50,
       verbose=False)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1000, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [13]:
pred = reg.predict(X_test)

In [14]:
pred

array([4477.8896, 4903.526 , 5487.1084, ..., 5085.2656, 4806.0835,
       4746.7427], shape=(19268,), dtype=float32)

In [ ]:
anomaly_detector = AnomalyDetection(pred, target_val) #

## Results and Analysis

In [15]:
#@markdown plot prediction
from plotly import graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=test_idx,
    y=y_test.to_numpy(),
    name='Actual',
    line_color='blue')
)

fig.add_trace(go.Scattergl(
    x=test_idx,
    y=pred,
    name='Predicted',
    line_color='red')
)


# Set the theme to 'plotly_white'
fig.update_layout(
    title=f"Time Series Forecasting for {target} with XGBoostRegressor",
    xaxis_title="t (1 unit = 1 hour)",
    yaxis_title="Energy Demand",
    template="plotly_white",
    xaxis = dict( rangeslider=dict(
      visible=True
    ))
)
fig.show()
fig.write_html(f'./outputs/xgb_mt1r1_pred.html')

In [16]:
#@markdown calculate MSE, MAE, MAPE
mse = mean_squared_error(y_true=y_test,
                   y_pred=pred)

mae = mean_absolute_error(y_true=y_test,
                   y_pred=pred)

def mean_absolute_percentage_error(y_true, y_pred):
    """Calculates MAPE given y_true and y_pred"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"RMSE: {np.sqrt(mse)}")
print(f"MAE: {mae}")
print(f"MAPE: {mean_absolute_percentage_error(y_test, pred)}")

RMSE: 462.4968918814482
MAE: 357.80401611328125
MAPE: 6.42711990756266


In [17]:
#@markdown top 10 worst predictions
tmp = pd.concat([dt, y_test, pd.Series(pred, index=y_test.index, name='pred')],
                axis=1).dropna()

tmp['error'] = tmp[target] - tmp['pred']
tmp['abs_error'] = tmp['error'].apply(np.abs)

worst_predicted = tmp.sort_values(by='abs_error', ascending=False)
worst_predicted[:10]

NameError: name 'dt' is not defined

In [14]:
#@markdown top 10 best predictions
best_predicted = tmp.sort_values(by='abs_error', ascending=True)
best_predicted[:10]

,DateTime,Toronto,pred,error,abs_error
96186,2020-12-21 19:00:00,6643.0,6643.044922,-0.044922,0.044922
77431,2018-11-01 08:00:00,5835.0,5834.920898,0.079102,0.079102
84910,2019-09-08 23:00:00,4496.0,4496.089844,-0.089844,0.089844
86672,2019-11-21 09:00:00,6105.0,6105.141113,-0.141113,0.141113
92354,2020-07-15 03:00:00,4881.0,4881.197754,-0.197754,0.197754
87050,2019-12-07 03:00:00,4852.0,4852.218262,-0.218262,0.218262
84429,2019-08-19 22:00:00,6568.0,6567.772461,0.227539,0.227539
89232,2020-03-07 01:00:00,5087.0,5086.763184,0.236816,0.236816
79964,2019-02-14 21:00:00,6621.0,6620.723633,0.276367,0.276367
95832,2020-12-07 01:00:00,4965.0,4965.280762,-0.280762,0.280762


In [15]:
best_predicted.iloc[:168].sort_index()

,DateTime,Toronto,pred,error,abs_error
77267,2018-10-25 12:00:00,5934.0,5935.841797,-1.841797,1.841797
77273,2018-10-25 18:00:00,6201.0,6202.523926,-1.523926,1.523926
77431,2018-11-01 08:00:00,5835.0,5834.920898,0.079102,0.079102
77437,2018-11-01 14:00:00,6052.0,6053.271973,-1.271973,1.271973
77459,2018-11-02 12:00:00,6145.0,6148.050293,-3.050293,3.050293
...,...,...,...,...,...
95868,2020-12-08 13:00:00,6358.0,6356.591309,1.408691,1.408691
95891,2020-12-09 12:00:00,6595.0,6592.875000,2.125000,2.125000
96178,2020-12-21 11:00:00,6229.0,6225.574219,3.425781,3.425781
96186,2020-12-21 19:00:00,6643.0,6643.044922,-0.044922,0.044922


In [24]:
#@markdown plot best/worst predictions
from plotly.subplots import make_subplots

hours = 168*10 # hours in a week
sorted_best_predicted = best_predicted.iloc[:hours].sort_index()
sorted_worst_predicted = worst_predicted.iloc[:hours].sort_index()


fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=(f'Best Predicted Hours for {target}',f'Worst Predicted Hours for {target}'))

# best predicted
fig.append_trace(go.Scattergl(
    x=sorted_best_predicted.DateTime,
    y=sorted_best_predicted[target],
    name='Target',
    mode='markers',
    # line_color='blue'
), row=1, col=1)

fig.append_trace(go.Scattergl(
    x=sorted_best_predicted.DateTime,
    y=sorted_best_predicted[target],
    name='Predicted',
    line_color='red'
), row=1, col=1)


# worst predicted
fig.append_trace(go.Scattergl(
    x=sorted_worst_predicted.DateTime,
    y=sorted_worst_predicted[target],
    name='Target',
    mode='markers',
    # line_color='blue'
), row=2, col=1)

fig.append_trace(go.Scattergl(
    x=sorted_worst_predicted.DateTime,
    y=sorted_worst_predicted.pred,
    name='Predicted',
    line_color='red'
), row=2, col=1)


fig.update_layout(
    title=f"Best/Worst Predictions for: {target}",

    # yaxis_title="Energy Demand",
    template="plotly_white",
)

fig.update_xaxes(
    title="t (1 unit = 1 hour)",
    rangeslider_visible=True, row=2, col=1)


# https://stackoverflow.com/questions/77370313/plotting-subplots-with-a-shared-slider-in-python
fig.show()
fig.write_html(f'./outputs/xgb_mt1r1_pred_eval.html')


## Wrapping Up

In [23]:
# save as csv for anomaly detection
tmp.to_csv('./outputs/xgb_mt1r1_pred_eval.csv')

In [22]:
# save learned model
reg.save_model('./outputs/xgb_mt1r1_model.json')  # Save model in JSON format

In [18]:
#@markdown wandb code for later

# !pip install wandb
# wandb login

# wandb.init(
#     # set the wandb project where this run will be logged
#     project="my-awesome-project",

#     # track hyperparameters and run metadata
#     config={
#     "learning_rate": 0.02,
#     "architecture": "CNN",
#     "dataset": "CIFAR-100",
#     "epochs": 10,
#     }
# )

# # simulate training
# epochs = 10
# offset = random.random() / 5
# for epoch in range(2, epochs):
#     acc = 1 - 2 ** -epoch - random.random() / epoch - offset
#     loss = 2 ** -epoch + random.random() / epoch + offset

#     # log metrics to wandb
#     wandb.log({"acc": acc, "loss": loss})

# # [optional] finish the wandb run, necessary in notebooks
# wandb.finish()